In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

# Text Preprocessing



In [7]:
# Converting lower case

df["review"] =  df["review"].str.lower()

In [8]:
# Removing the URLs

import re

def remove_urls(text):
    re.sub(r"http\S+", "", text)
    return text
df["review"] =  df["review"].apply(remove_urls)

In [9]:
# Removing Punctuation
def remove_punctuations(text):
    text = re.sub(r"^A-Za-z0-9\s]", "", text)
    return text

df["review"] =  df["review"].apply(remove_punctuations)

In [10]:
# Removing HTML

def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] =  df["review"].apply(remove_html)


In [12]:
# Removing the StopWords

import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Ainvio\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Ainvio\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ainvio\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [14]:
def remove_stopword(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    return text

df["review"] =  df["review"].apply(remove_stopword)

In [15]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode 'll h...,positive
1,wderful ltle producti. filming technique u...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly 's fmly lttle boy (jke) thks 's zom...,negative
4,"petter mttei's ""love time mey"" vully stun...",positive


In [17]:
# Stemming (running --> run, played --> play)

from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)
    
df["review"] =  df["review"].apply(stemming)
    

In [18]:
# Encoding for target value

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [19]:
y = df["sentiment"]

In [20]:
y 

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [21]:
# Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X= tf.fit_transform(df["review"])
 

In [23]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4093502 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3547)	0.05259261191241623
  (0, 2874)	0.0891811480809547
  (0, 4944)	0.11139769507366715
  (0, 3006)	0.5407847094408476
  (0, 1281)	0.13101059007341456
  (0, 2286)	0.09233878370751725
  (0, 1928)	0.07545002362581753
  (0, 3560)	0.09118781496658057
  (0, 1365)	0.059149764165192874
  (0, 1959)	0.05884977654648405
  (0, 1619)	0.07013080150908861
  (0, 4375)	0.041140089431941564
  (0, 4176)	0.16983030244241037
  (0, 3702)	0.032031116916697626
  (0, 4745)	0.25491508883451863
  (0, 3811)	0.04196892128560127
  (0, 4778)	0.11153082068688318
  (0, 1739)	0.03547069573441665
  (0, 4501)	0.07283478628502871
  (0, 3862)	0.16752441858895883
  (0, 1629)	0.0576211701071577
  (0, 1859)	0.06845754722083207
  (0, 3338)	0.06136616893457463
  (0, 3341)	0.08006306910104358
  (0, 3442)	0.07509430628965053
  :	:
  (49581, 1843)	0.05319668362982228
  (49581, 2767)	0.0701565747061

In [25]:
# Dataset and Data Loaders
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [26]:
X_train.shape

(39665, 5000)

In [27]:
X_test.shape

(9917, 5000)

In [28]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [29]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [30]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [31]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(train_set, shuffle=True, batch_size=64)

# Build RNN

In [34]:
import torch.nn as nn
import torch.optim as optim


In [35]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        #optional --> shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        #1sr value = hiddent state of all the timesteps --> (batch, seq_len, hidden size)
        #2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

        

In [37]:
input_size = X_train.shape[1]

model =  RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Training RNN

In [39]:
# unsqueeze squeeze

epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)  # add singleton direction

        outputs = model(Xb) # batch_size, 1

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) --> probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward()

        optimizer.step() # weight update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.210083469748497
epoch = 2/10 and loss = 0.22599025070667267
epoch = 3/10 and loss = 0.33033713698387146
epoch = 4/10 and loss = 0.19506341218948364
epoch = 5/10 and loss = 0.2226681262254715
epoch = 6/10 and loss = 0.18314769864082336
epoch = 7/10 and loss = 0.2765653133392334
epoch = 8/10 and loss = 0.2704373896121979
epoch = 9/10 and loss = 0.1965191513299942
epoch = 10/10 and loss = 0.12867645919322968


In [41]:
# Evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())> 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb ).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}%")

accuracy = 91.81394176225892%
